In [5]:
import tensorflow as tf
import tensorflow.keras as keras
from keras import layers
import numpy as np
import deeplake as dl

c:\Users\etito\miniconda3\Lib\site-packages\deeplake\util\check_latest_version.py:32: UserWarning: A newer version of deeplake (4.6.3) is available. It's recommended that you update to the latest version using `pip install -U deeplake`.
  warnings.warn(


In [6]:
# Loading the Dataset
train_ds = dl.load("hub://activeloop/visdrone-det-train")
val_ds = dl.load("hub://activeloop/visdrone-det-val")
print(train_ds.summary())

|

Opening dataset in read-only mode as you don't have write permissions.


/

This dataset can be visualized in Jupyter Notebook by ds.visualize() or at https://app.activeloop.ai/activeloop/visdrone-det-train



|

...

Opening dataset in read-only mode as you don't have write permissions.


-

This dataset can be visualized in Jupyter Notebook by ds.visualize() or at https://app.activeloop.ai/activeloop/visdrone-det-val



/

hub://activeloop/visdrone-det-val loaded successfully.

Dataset(path='hub://activeloop/visdrone-det-train', read_only=True, tensors=['boxes', 'images', 'labels'])

 tensor      htype                 shape               dtype  compression
 -------    -------               -------             -------  ------- 
  boxes      bbox            (6471, 1:914, 4)         float32   None   
 images      image     (6471, 360:1500, 480:2000, 3)   uint8    jpeg   
 labels   class_label          (6471, 1:914)          uint32    None   
None


In [7]:
# Preprocessing the images
train_stream = train_ds.tensorflow()
val_stream = val_ds.tensorflow()
BATCH_SIZE = 32
AUTOTUNE = tf.data.AUTOTUNE

def image_processing(data):
    # Extract the image and resize it to a shape still
    image = data['images']

    image = tf.image.resize(image, (360, 480))
    image = tf.cast(image, tf.float32) / 255.0
    
    return image

train_dataset = (train_stream
                 .map(image_processing, num_parallel_calls=AUTOTUNE)
                 .batch(BATCH_SIZE)
                 .prefetch(AUTOTUNE))

val_dataset = (val_stream
               .map(image_processing, num_parallel_calls=AUTOTUNE)
               .batch(BATCH_SIZE)
               .prefetch(AUTOTUNE))

# Preprocessing the city labels
def process_labels(city_data):
    city_label_to_index = {label: index for index, label in enumerate(city_data)}
    numerical_labels = np.array([city_label_to_index[name] for name in city_data], dtype=np.int32)
    return numerical_labels

city_labels = ["tianjin", "hong-kong", "daqing", "ganzhou", 
               "guangzhou", "jinchang", "liuzhou", "nanjing", 
               "shaoxing", "shenyang", "nanyang", "zhangjiakou", 
               "suzhou", "xuzhou"]

train_labels = process_labels(city_labels)

In [8]:
# Building the Model
def build_multi_head_model():
    input_tensor = layers.Input(shape=(360, 480, 3), name='input_image')

    base_model = keras.applications.ResNet50(
        weights='imagenet', 
        include_top=False, 
        input_tensor=input_tensor
    )
    base_model.trainable = False

    shared_features = base_model.output

    # 1. Object Detection Head (Processes backbone features first)
    x_detect = layers.GlobalAveragePooling2D()(shared_features)
    x_detect = layers.Dense(128, activation='relu')(x_detect)
    bbox_output = layers.Dense(4, activation='sigmoid', name="bbox_output")(x_detect)

    # 2. City Classification Head (Dependent on Detection)
    x_class_input = layers.GlobalAveragePooling2D()(shared_features)
    
    # Concatenate blends the spatial context with the localization features
    merged_features = layers.Concatenate()([x_class_input, x_detect])
    
    x_class = layers.Dense(256, activation='relu')(merged_features)
    x_class = layers.Dropout(0.5)(x_class)
    city_output = layers.Dense(14, activation='softmax', name="city_output")(x_class)

    model = keras.Model(inputs=input_tensor, outputs=[city_output, bbox_output])

    lr_scheduler = keras.optimizers.schedules.CosineDecay(
        initial_learning_rate=0.001,
        decay_steps=10000,
        alpha=0.0
    )
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=lr_scheduler),
        loss={
            "city_output": "sparse_categorical_crossentropy",
            "bbox_output": "mean_squared_error"
        },
        metrics={
            "city_output": "accuracy",
            "bbox_output": "mae"
        },
        loss_weights={
            "city_output": 1.0, 
            "bbox_output": 2.0
        }
    )
    return model
china_model = build_multi_head_model()

94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 4s 0us/step
